#Explanation


The `compute_rule_score` function, defined in the cell above, checks for several characteristics commonly associated with phishing URLs. It assigns a point for each rule triggered, and then normalizes this score to be between 0 and 1. Here are the rules and the parameters it uses:

*   **`SUSPICIOUS_TLDS`**: A list of top-level domains (like `.tk`, `.ml`, `.zip`) that are frequently abused by phishers. If a URL uses one of these, it triggers a rule.
*   **`PHISHING_KEYWORDS`**: Common keywords (like `login`, `verify`, `account`) found in phishing URLs that attempt to deceive users. If any of these keywords are present in the URL, a rule is triggered.
*   **`URL_SHORTENERS`**: Checks if the URL uses a known URL shortening service (like `bit.ly`, `tinyurl`). Phishers often use these to hide the true destination of a malicious link.
*   **`@` symbol**: The presence of an `@` symbol in a URL can indicate an attempt to embed credentials or confuse users about the true domain.
*   **Excessive Subdomains**: URLs with more than 4 dots (e.g., `very.long.subdomain.example.com`) can sometimes be a sign of obfuscation.

The `compute_rule_score` function is then applied to each URL in the dataset. This creates two new columns:

*   **`rule_score`**: A normalized score (0 to 1) indicating how many rules the URL triggered.
*   **`triggered_rules`**: A list of the specific rules that were triggered for each URL.

Finally, this `rule_score` is integrated with the machine learning model's prediction to calculate a combined `Trust_Index`. A higher `rule_score` (meaning more rules were triggered) will generally decrease the `Trust_Index`, indicating a higher likelihood of the URL being malicious, even if the ML model's probability is low. This creates a robust hybrid detection system.

##Evaluation Metrics

1.  Accuracy: 0.9947
2.  Precision: 0.9951
3.  Recall: 0.9944
4.  F1-Score: 0.9947

#S1

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


##Part -1 Environment

In [1]:
!python --version

Python 3.12.13


In [2]:
!pip install --upgrade pip
!pip install pandas numpy scikit-learn matplotlib seaborn tqdm sentence-transformers tldextract requests joblib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 22.7 MB/s eta 0:00:00
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 447, in run
    conflicts = self._determine_conflicts(to_install)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 578, in _determine_conflicts
    return check_install_conflicts(to_install)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/operations/check.py", line 101, in check_install_conf

In [3]:
import pandas, numpy, sklearn, matplotlib, seaborn, tldextract, requests, joblib
print("✅ All libraries imported successfully!")


✅ All libraries imported successfully!


In [4]:
!cat /proc/cpuinfo | grep "model name"

model name	: Intel(R) Xeon(R) CPU @ 2.20GHz
model name	: Intel(R) Xeon(R) CPU @ 2.20GHz


In [5]:
import os
from google.colab import drive

# Ensure the mount point is truly empty
if os.path.exists('/content/drive'):
    # Attempt to unmount first if it's already mounted
    try:
        !fusermount -uz /content/drive
    except:
        pass # Ignore if not mounted or unmounting fails
    !rm -rf /content/drive
os.makedirs('/content/drive', exist_ok=True)

drive.mount('/content/drive', force_remount=True)

MessageError: Error: credential propagation was unsuccessful

In [ ]:
import os
# Corrected base_path to be a directory where project files will be stored
base_path = "/content/drive/MyDrive/Phishing_Project"
os.makedirs(base_path, exist_ok=True)
print("Working folder:", base_path)

In [ ]:
folders = ["data", "models", "outputs"]
for f in folders:
    os.makedirs(f"{base_path}/{f}", exist_ok=True)
print("✅ Folder structure ready:", folders)

In [ ]:
import json
config = {
    "base_path": base_path,
    "dataset_name": "phish_dataset.csv",
    "random_state": 42
}
json.dump(config, open(f"{base_path}/config.json", "w"))
print("✅ Config file saved!")


## Part-2 Obtaining DataSet and Cleaning

In [ ]:
!pip install kagglehub pandas numpy matplotlib seaborn scikit-learn tqdm tldextract


In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("shashwatwork/web-page-phishing-detection-dataset")

print("Path to dataset files:", path)

In [ ]:
import os
os.listdir(path)


In [ ]:
import pandas as pd

file_path = os.path.join(path, "dataset_phishing.csv")   # adjust if different
df = pd.read_csv(file_path)
print("✅ Dataset loaded. Rows:", len(df))
display(df.head())

In [ ]:
df.info()
df.sample(5)

In [ ]:
# Show all column names to identify what the label column is called
print("Columns found:", df.columns.tolist())
df.head(3)


In [ ]:
# Rename correct columns
df.rename(columns={'status': 'label'}, inplace=True)

# Keep URL and label for reference, but we’ll keep all features too
print("✅ Columns standardized successfully!")

# Convert labels to numeric (phishing = 1, legitimate = 0)
df['label'] = df['label'].replace({'legitimate': 0, 'benign': 0, 'phishing': 1, 'malicious': 1}).astype(int)

print("✅ Label distribution:\n", df['label'].value_counts())
df.head(3)


In [ ]:
# Drop NaNs or blanks
df = df.dropna(subset=['url'])

# Remove duplicates
df = df.drop_duplicates(subset=['url'])

# Convert labels to 0/1 integers
df['label'] = df['label'].replace({'legitimate': 0, 'benign': 0, 'phishing': 1, 'malicious': 1}).astype(int)

print("✅ Clean dataset shape:", df.shape)
display(df['label'].value_counts())

In [ ]:
#Normalizinng the URL text

df['url'] = df['url'].astype(str).str.strip().str.lower()


In [ ]:
#Visualize Class Balance
import matplotlib.pyplot as plt
plt.figure(figsize=(4,3))
df['label'].value_counts().plot(kind='bar', color=['skyblue','salmon'])
plt.xticks([0,1], ['Legitimate (0)','Phishing (1)'], rotation=0)
plt.title("Dataset Balance")
plt.show()


In [ ]:
from sklearn.utils import resample

phish = df[df['label'] == 1]
legit = df[df['label'] == 0]

min_size = min(len(phish), len(legit))

df_bal = pd.concat([
    resample(phish, n_samples=min_size, random_state=42),
    resample(legit, n_samples=min_size, random_state=42)
]).sample(frac=1, random_state=42).reset_index(drop=True)

print("✅ Balanced dataset shape:", df_bal.shape)
df = df_bal

In [ ]:
base_path = "/content/drive/MyDrive/Phishing_Project"
os.makedirs(f"{base_path}/data", exist_ok=True)

df.to_csv(f"{base_path}/data/phish_dataset.csv", index=False)
print("✅ Clean dataset saved at:", f"{base_path}/data/phish_dataset.csv")


## Part- 3 Feature Engineering

In [ ]:
feature_cols = [c for c in df.columns if c not in ['url','label']]
print("Total numeric features:", len(feature_cols))
print("First 10:", feature_cols[:10])

In [ ]:
import numpy as np

X = df[feature_cols].to_numpy()
# X=X[:,1:]
y = df['label'].to_numpy()

print("✅ Feature matrix shape:", X.shape)
print("✅ Labels shape:", y.shape)


In [ ]:
#scale features

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("✅ Features standardized.")

In [ ]:
#Feature importance check

from sklearn.ensemble import RandomForestClassifier
import pandas as pd

rf_temp = RandomForestClassifier(n_estimators=100, random_state=42)
rf_temp.fit(X_scaled, y)

importances = pd.Series(rf_temp.feature_importances_, index=feature_cols)
top10 = importances.sort_values(ascending=False)[:10]

print("Top 10 important features:\n", top10)


In [ ]:
#plot
import matplotlib.pyplot as plt
top10.plot(kind='barh', color='teal')
plt.title("Top 10 Feature Importances (Quick RF Test)")
plt.gca().invert_yaxis()
plt.show()


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.25, random_state=42, stratify=y
)

print("✅ Training samples:", X_train.shape[0])
print("✅ Test samples:", X_test.shape[0])


In [ ]:
import joblib, os

base_path = "/content/drive/MyDrive/Phishing_Project"
os.makedirs(f"{base_path}/data", exist_ok=True)

np.save(f"{base_path}/data/X_train.npy", X_train)
np.save(f"{base_path}/data/X_test.npy", X_test)
np.save(f"{base_path}/data/y_train.npy", y_train)
np.save(f"{base_path}/data/y_test.npy", y_test)
joblib.dump(scaler, f"{base_path}/data/scaler.pkl")

print("✅ Processed data saved to Drive.")


## Part-3.5: Advanced Feature Selection

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_selection import SelectKBest, f_classif, mutual_info_classif, RFE
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
import warnings
warnings.filterwarnings('ignore')

In [ ]:
feature_cols = [c for c in df.columns if c not in ['url','label']]
print("Total numeric features:", len(feature_cols))
print("First 10:", feature_cols[:10])

In [ ]:
print("✅ Feature matrix shape:", X_scaled.shape)
print("✅ Labels shape:", y.shape)
print(len(feature_cols))

### 1. Correlation Analysis with Target

In [ ]:
print("\n1️⃣ Correlation Analysis with Target")
correlations = pd.DataFrame({
    'feature': feature_cols[1:-1],
    'correlation': [np.corrcoef(X_scaled[:, i], y)[0, 1] for i in range(len(feature_cols)-2)]
})
correlations['abs_correlation'] = abs(correlations['correlation'])
correlations = correlations.sort_values('abs_correlation', ascending=False)

# Display top 20 correlated features
print("\nTop 20 features by absolute correlation with target:")
display(correlations.head(20))

In [ ]:
plt.figure(figsize=(12, 10))
top_features = correlations.head(20)['feature'].tolist()
top_indices = [feature_cols.index(f) for f in top_features]
sns.heatmap(pd.DataFrame(X_scaled[:, top_indices]).corr(),
            annot=True, fmt='.2f', cmap='coolwarm',
            xticklabels=top_features, yticklabels=top_features)
plt.title('Correlation Heatmap - Top 20 Features')
plt.tight_layout()
plt.show()

### 2. Univariate Feature Selection (ANOVA F-test)

In [ ]:
print("\n2️⃣ Univariate Feature Selection (ANOVA F-test)")
selector_f = SelectKBest(score_func=f_classif, k='all')
selector_f.fit(X_scaled, y)
f_scores = pd.DataFrame({
    'feature': feature_cols,
    'f_score': selector_f.scores_,
    'p_value': selector_f.pvalues_
}).sort_values('f_score', ascending=False)

print("\nTop 20 features by F-score:")
display(f_scores.head(20))

# Plot F-scores
plt.figure(figsize=(10, 6))
plt.barh(f_scores.head(20)['feature'], f_scores.head(20)['f_score'], color='steelblue')
plt.xlabel('F-score')
plt.title('Top 20 Features by ANOVA F-score')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

### 3. Mutual Information

In [ ]:
mi_scores = mutual_info_classif(X_scaled, y, random_state=42)
mi_df = pd.DataFrame({
    'feature': feature_cols,
    'mi_score': mi_scores
}).sort_values('mi_score', ascending=False)

print("\nTop 20 features by Mutual Information:")
display(mi_df.head(20))

# Plot MI scores
plt.figure(figsize=(10, 6))
plt.barh(mi_df.head(20)['feature'], mi_df.head(20)['mi_score'], color='coral')
plt.xlabel('Mutual Information Score')
plt.title('Top 20 Features by Mutual Information')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

### 4. Recursive Feature Elimination (RFE) with Random Forest

In [ ]:
rfe_estimator = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rfe = RFE(estimator=rfe_estimator, n_features_to_select=50, step=5, verbose=1)
rfe.fit(X_scaled, y)

rfe_features = pd.DataFrame({
    'feature': feature_cols,
    'selected': rfe.support_,
    'ranking': rfe.ranking_
}).sort_values('ranking')

print(f"\nRFE selected {rfe.support_.sum()} features")
print("\nTop 20 features by RFE ranking:")
display(rfe_features.head(20))


### 5. Random Forest Feature Importance (with cross-validation)

In [ ]:
rf_importance = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf_importance.fit(X_scaled, y)

importance_df = pd.DataFrame({
    'feature': feature_cols,
    'importance': rf_importance.feature_importances_
}).sort_values('importance', ascending=False)

print("\nTop 20 features by Random Forest importance:")
display(importance_df.head(20))

# Plot RF importance
plt.figure(figsize=(10, 6))
plt.barh(importance_df.head(20)['feature'], importance_df.head(20)['importance'], color='forestgreen')
plt.xlabel('Feature Importance')
plt.title('Top 20 Features - Random Forest Importance')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

### 6. Combined Ranking (Ensemble Feature Selection)

In [ ]:
combined_scores = pd.DataFrame({'feature': feature_cols})

# Normalize each score to 0-1 range
def normalize_scores(scores):
    # Handle case where all scores are the same (e.g., all 0 or all 1) to avoid division by zero
    if scores.max() == scores.min():
        return pd.Series([0.0] * len(scores), index=scores.index) # Return all zeros if no variation
    return (scores - scores.min()) / (scores.max() - scores.min())

# Merge correlation scores
correlations_to_merge = correlations[['feature', 'abs_correlation']].rename(columns={'abs_correlation': 'correlation_score_raw'})
combined_scores = pd.merge(combined_scores, correlations_to_merge, on='feature', how='left')

# Merge F-scores
f_scores_to_merge = f_scores[['feature', 'f_score']]
combined_scores = pd.merge(combined_scores, f_scores_to_merge, on='feature', how='left')

# Merge MI scores
mi_df_to_merge = mi_df[['feature', 'mi_score']]
combined_scores = pd.merge(combined_scores, mi_df_to_merge, on='feature', how='left')

# Merge RFE ranks
rfe_features['rfe_score_raw'] = 1 / rfe_features['ranking'] # Lower ranking is better
rfe_features_to_merge = rfe_features[['feature', 'rfe_score_raw']]
combined_scores = pd.merge(combined_scores, rfe_features_to_merge, on='feature', how='left')

# Merge RF importance
importance_df_to_merge = importance_df[['feature', 'importance']].rename(columns={'importance': 'rf_importance_raw'})
combined_scores = pd.merge(combined_scores, importance_df_to_merge, on='feature', how='left')

# Fill NaN values with 0 for scores where a feature might have been missing in an original calculation
combined_scores = combined_scores.fillna(0)

# Apply normalization to the raw scores
combined_scores['correlation_norm'] = normalize_scores(combined_scores['correlation_score_raw'])
combined_scores['f_score_norm'] = normalize_scores(combined_scores['f_score'])
combined_scores['mi_score_norm'] = normalize_scores(combined_scores['mi_score'])
combined_scores['rfe_rank_norm'] = normalize_scores(combined_scores['rfe_score_raw'])
combined_scores['rf_importance_norm'] = normalize_scores(combined_scores['rf_importance_raw'])

# Calculate combined score (average of all normalized scores)
combined_scores['combined_score'] = combined_scores[[
    'correlation_norm', 'f_score_norm', 'mi_score_norm', 'rfe_rank_norm', 'rf_importance_norm'
]].mean(axis=1)
combined_scores = combined_scores.sort_values('combined_score', ascending=False)

print("\nTop 20 features by combined ranking:")
display(combined_scores.head(20))

# Plot combined scores
plt.figure(figsize=(10, 6))
plt.barh(combined_scores.head(20)['feature'], combined_scores.head(20)['combined_score'], color='purple')
plt.xlabel('Combined Score')
plt.title('Top 20 Features - Ensemble Selection')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

### 7. Select optimal number of features using cross-validation

In [ ]:
# Test different numbers of features
k_values = [10, 20, 30, 40, 50, 75, 100, 150, 200]
cv_scores = []

# Use top features from combined ranking
top_features_by_rank = combined_scores['feature'].tolist()

for k in k_values:
    # Select top k features
    selected_indices = [feature_cols.index(f) for f in top_features_by_rank[:k]]
    X_selected = X_scaled[:, selected_indices]

    # Cross-validation score
    rf_temp = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    cv_score = cross_val_score(rf_temp, X_selected, y, cv=5, scoring='accuracy').mean()
    cv_scores.append(cv_score)
    print(f"k={k:3d}, CV Accuracy={cv_score:.4f}")

# Plot CV scores
plt.figure(figsize=(10, 6))
plt.plot(k_values, cv_scores, 'bo-', linewidth=2, markersize=8)
plt.xlabel('Number of Features (k)')
plt.ylabel('Cross-Validation Accuracy')
plt.title('Optimal Number of Features Selection')
plt.grid(True, alpha=0.3)
plt.xticks(k_values)
plt.tight_layout()
plt.show()

# Find optimal k
optimal_k = k_values[np.argmax(cv_scores)]
print(f"\n✅ Optimal number of features: {optimal_k} (CV Accuracy: {max(cv_scores):.4f})")

### 8. Final feature selection

In [ ]:
selected_features = top_features_by_rank[:optimal_k]
selected_indices = [feature_cols.index(f) for f in selected_features]

print(f"\nSelected {len(selected_features)} features:")
for i, feat in enumerate(selected_features[:30], 1):
    print(f"{i:2d}. {feat}")
if len(selected_features) > 30:
    print(f"... and {len(selected_features)-30} more")

# Create reduced dataset
X_selected_final = X_scaled[:, selected_indices]
print(f"\n✅ Reduced feature matrix shape: {X_selected_final.shape}")

### 9. Evaluate model with selected features

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Split data
X_train_sel, X_test_sel, y_train_sel, y_test_sel = train_test_split(
    X_selected_final, y, test_size=0.25, random_state=42, stratify=y
)

# Train Random Forest on selected features
rf_selected = RandomForestClassifier(n_estimators=200, max_depth=15, random_state=42, n_jobs=-1)
rf_selected.fit(X_train_sel, y_train_sel)
y_pred_sel = rf_selected.predict(X_test_sel)

# Evaluate model with selected features
print(f"\nAccuracy with selected features: {accuracy_score(y_test_sel, y_pred_sel):.4f}")
print("\nClassification Report:")
print(classification_report(y_test_sel, y_pred_sel))

# To compare with the original model, we need to train the original model first.
# X_train, X_test, y_train, y_test are available from a previous step (split of X_scaled and y).
rf_original = RandomForestClassifier(n_estimators=200, max_depth=15, random_state=42, n_jobs=-1)
rf_original.fit(X_train, y_train) # Using the full feature set splits
pred_rf = rf_original.predict(X_test) # Predictions from the original model

# Compare with original model
print("\n📊 Comparison with original model (all features):")
print(f"Original model accuracy: {accuracy_score(y_test, pred_rf):.4f}")
print(f"Selected features model accuracy: {accuracy_score(y_test_sel, y_pred_sel):.4f}")
print(f"Feature reduction: {X_scaled.shape[1]} → {X_selected_final.shape[1]} features")
print(f"Reduction percentage: {(1 - X_selected_final.shape[1]/X_scaled.shape[1])*100:.1f}%")

### 10. Save selected features and model

In [ ]:
# Save selected feature names
selected_features_df = pd.DataFrame({
    'selected_features': selected_features,
    'ranking': range(1, len(selected_features)+1)
})
selected_features_df.to_csv(f"{base_path}/selected_features.csv", index=False)
print(f"✅ Saved selected features to {base_path}/selected_features.csv")

# Save the reduced datasets
np.save(f"{base_path}/X_train_selected.npy", X_train_sel)
np.save(f"{base_path}/X_test_selected.npy", X_test_sel)
np.save(f"{base_path}/y_train_selected.npy", y_train_sel)
np.save(f"{base_path}/y_test_selected.npy", y_test_sel)

# Save the selected features model
joblib.dump(rf_selected, f"{base_path}/rf_model_selected.pkl")
print(f"✅ Saved model with selected features to {base_path}/rf_model_selected.pkl")

# Save feature selector information
selector_info = {
    'optimal_k': optimal_k,
    'selected_features': selected_features,
    'original_feature_count': X_scaled.shape[1],
    'selected_feature_count': X_selected_final.shape[1],
    'cv_scores': cv_scores,
    'k_values': k_values
}
joblib.dump(selector_info, f"{base_path}/feature_selector_info.pkl")
print("✅ Saved feature selector information")

print("\n" + "="*50)
print("✅ FEATURE SELECTION COMPLETE!")
print("="*50)

##Part-3.6 actual feature reducton

### Checking

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_selection import SelectKBest, f_classif, mutual_info_classif, RFE, VarianceThreshold
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, train_test_split
import warnings
warnings.filterwarnings('ignore')

print("="*60)
print("DEBUGGING: WHAT'S HAPPENING WITH YOUR FEATURES")
print("="*60)

# Check what features you actually have
print(f"\n📊 Original feature count: {len(feature_cols)}")
print(f"First 10 features: {feature_cols[:10]}")

# Check for zero-variance features (completely useless)
variance_selector = VarianceThreshold(threshold=0.01)  # Remove features with very low variance
variance_selector.fit(X_scaled)
low_variance_mask = variance_selector.variances_ < 0.01
low_variance_features = [feature_cols[i] for i, mask in enumerate(low_variance_mask) if mask]

print(f"\n🔍 Low variance features (threshold < 0.01): {len(low_variance_features)}")
if len(low_variance_features) > 0:
    print(f"   Examples: {low_variance_features[:5]}")

# Check for highly correlated features
correlation_matrix = pd.DataFrame(X_scaled).corr().abs()
upper_tri = correlation_matrix.where(np.triu(np.ones(correlation_matrix.shape), k=1).astype(bool))
high_corr_pairs = [(feature_cols[i], feature_cols[j], upper_tri.iloc[i, j])
                   for i in range(len(feature_cols))
                   for j in range(len(feature_cols))
                   if upper_tri.iloc[i, j] > 0.95]

print(f"\n🔗 Highly correlated feature pairs (>0.95): {len(high_corr_pairs)}")
if len(high_corr_pairs) > 0:
    for i, (f1, f2, corr) in enumerate(high_corr_pairs[:5]):
        print(f"   {i+1}. {f1} ↔ {f2}: {corr:.3f}")

print("\n" + "="*60)
print("PROPER FEATURE SELECTION WITH AGGRESSIVE REDUCTION")
print("="*60)

##Part - 4 Feature extraction


* **ip**: Indicates whether the URL contains an IP address instead of a domain name, which is often suspicious.
* **nb_qm**: The number of question marks (?) in the URL, often used to append parameters or obfuscate.
* **nb_www**: The presence of "www" in the URL, which can sometimes be normal but also used as a phishing trick.
* **ratio_digits_url**: The ratio of digits to the total length of the URL. High digit ratios can indicate generated or suspicious URLs.
* **phish_hints**: Indicates the presence of phishing-related keywords or patterns derived from the dataset.
* **nb_hyperlinks**: The number of hyperlinks in the page content; higher values may indicate redirects or malicious structure.
* **domain_in_title**: Checks if the domain name appears in the HTML page title. Legitimate sites usually include it.
* **domain_age**: The age of the domain in days. Newer domains are often associated with phishing.
* **google_index**: Whether the URL is indexed by Google (1 = indexed, 0 = not indexed).
* **page_rank**: A measure of webpage authority; phishing sites typically have low values.

In [ ]:
df.info()

In [ ]:
df.loc[:, [c for c in df.columns if c not in ['url','label']]]

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif # Changed chi2 to f_classif to handle negative numbers in features
import numpy as np # Import numpy for nan_to_num

# Redefine feature_cols to exclude 'url' and 'label' columns
feature_cols_numeric = [c for c in df.columns if c not in ['url', 'label']]

# Prepare X and y with only numeric features
X = df[feature_cols_numeric].to_numpy()
y = df['label'].to_numpy()

# Now apply SelectKBest
selector = SelectKBest(score_func=f_classif, k=10) # Changed score_func from chi2 to f_classif
X_new = selector.fit_transform(X, y)

print("Shape of new feature matrix:", X_new.shape)

# Get selected feature names
selected_features = np.array(feature_cols_numeric)[selector.get_support()]
print("Selected features:", selected_features.tolist())

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Calculate correlation with label for all numeric columns
# We exclude 'url' as it is non-numeric
numeric_df = df.select_dtypes(include=['number'])
all_correlations = numeric_df.corr()['label'].drop('label')

# Get absolute values to find strongest relationships regardless of direction
abs_correlations = all_correlations.abs().sort_values(ascending=False)

# Get top 20
top_20_features = abs_correlations.head(20)

# Display the raw values
print("Top 20 Features by Correlation Magnitude:")
display(all_correlations.loc[top_20_features.index])

# Visualize
plt.figure(figsize=(10, 8))
sns.barplot(x=top_20_features.values, y=top_20_features.index, palette='magma')
plt.title('Top 20 Features (Absolute Correlation with Label)')
plt.xlabel('Absolute Correlation Coefficient')
plt.ylabel('Feature Name')
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.show()

In [ ]:
import re
import requests
from urllib.parse import urlparse
from bs4 import BeautifulSoup
import socket

# Optional: install python-whois if using domain_age
# pip install python-whois
try:
    import whois
except:
    whois = None


PHISH_KEYWORDS = [
    "login", "verify", "update", "secure", "account",
    "bank", "signin", "confirm", "password"
]


def extract_top12_features(url):
    features = {}

    # ------------------------
    # URL parsing
    # ------------------------
    parsed = urlparse(url)
    hostname = parsed.netloc

    # 1. length_url
    features['length_url'] = len(url)

    # 2. length_hostname
    features['length_hostname'] = len(hostname)

    # 3. nb_www
    features['nb_www'] = url.count("www")

    # 4. nb_qm
    features['nb_qm'] = url.count("?")

    # 5. ratio_digits_url
    digits = sum(c.isdigit() for c in url)
    features['ratio_digits_url'] = digits / len(url) if len(url) > 0 else 0

    # 6. ip (check if hostname is IP)
    features['ip'] = 1 if re.match(r'\d+\.\d+\.\d+\.\d+', hostname) else 0

    # 7. phish_hints
    features['phish_hints'] = sum(1 for word in PHISH_KEYWORDS if word in url.lower())

    # ------------------------
    # Fetch webpage (optional)
    # ------------------------
    try:
        response = requests.get(url, timeout=5)
        html = response.text
        soup = BeautifulSoup(html, "html.parser")

        # 8. nb_hyperlinks
        links = soup.find_all("a")
        features['nb_hyperlinks'] = len(links)

        # 9. ratio_intHyperlinks
        internal_links = 0
        for link in links:
            href = link.get("href")
            if href and hostname in href:
                internal_links += 1

        features['ratio_intHyperlinks'] = (
            internal_links / len(links) if len(links) > 0 else 0
        )

        # 10. domain_in_title
        title = soup.title.string if soup.title else ""
        features['domain_in_title'] = 1 if hostname in title else 0

    except:
        features['nb_hyperlinks'] = 0
        features['ratio_intHyperlinks'] = 0
        features['domain_in_title'] = 0

    # ------------------------
    # 11. domain_age (optional)
    # ------------------------
    if whois:
        try:
            w = whois.whois(hostname)
            creation_date = w.creation_date

            if isinstance(creation_date, list):
                creation_date = creation_date[0]

            if creation_date:
                age_days = (datetime.now() - creation_date).days
                features['domain_age'] = age_days
            else:
                features['domain_age'] = 0
        except:
            features['domain_age'] = 0
    else:
        features['domain_age'] = 0

    # ------------------------
    # 12. google_index (mock)
    # ------------------------
    # Real Google index requires API/scraping
    # Placeholder: assume indexed if response OK
    features['google_index'] = 1 if 'response' in locals() and response.status_code == 200 else 0

    return features

## Part-5 Train and evaluate

In [ ]:
#Load Processed data

import numpy as np, joblib

base_path = "/content/drive/MyDrive/Phishing_Project/data"

X_train = np.load(f"{base_path}/X_train.npy")
X_test  = np.load(f"{base_path}/X_test.npy")
y_train = np.load(f"{base_path}/y_train.npy")
y_test  = np.load(f"{base_path}/y_test.npy")

scaler = joblib.load(f"{base_path}/scaler.pkl")

print("✅ Data loaded. Shapes:", X_train.shape, X_test.shape)


In [ ]:
#Train, evaluate and compare

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train, y_train)
pred_lr = lr.predict(X_test)

print("🔹 Logistic Regression Results 🔹")
print("Accuracy:", accuracy_score(y_test, pred_lr))
print(classification_report(y_test, pred_lr))

In [ ]:
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier(max_depth=10, random_state=42)
dt.fit(X_train, y_train)
pred_dt = dt.predict(X_test)

print("🔹 Decision Tree Results 🔹")
print("Accuracy:", accuracy_score(y_test, pred_dt))
print(classification_report(y_test, pred_dt))


In [ ]:
# Random Forest

from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=200, max_depth=15, random_state=42)
rf.fit(X_train, y_train)
pred_rf = rf.predict(X_test)

print("🔹 Random Forest Results 🔹")
print("Accuracy:", accuracy_score(y_test, pred_rf))
print(classification_report(y_test, pred_rf))


In [ ]:
# Confusion matrix
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

models = {'LogReg': lr, 'DecisionTree': dt, 'RandomForest': rf}
for name, model in models.items():
    ConfusionMatrixDisplay.from_estimator(model, X_test, y_test)
    plt.title(f"Confusion Matrix — {name}")
    plt.show()


In [ ]:
#Comparing models

import pandas as pd
from sklearn.metrics import precision_score, recall_score, f1_score

def evaluate(model, X, y):
    p = model.predict(X)
    return {
        "Accuracy": round(accuracy_score(y, p), 4),
        "Precision": round(precision_score(y, p), 4),
        "Recall": round(recall_score(y, p), 4),
        "F1": round(f1_score(y, p), 4)
    }

results = pd.DataFrame({
    "Logistic Regression": evaluate(lr, X_test, y_test),
    "Decision Tree": evaluate(dt, X_test, y_test),
    "Random Forest": evaluate(rf, X_test, y_test)
}).T

print(results)


In [ ]:
# saving the best moidel

import joblib, os

best_model = rf   # change if another performs better

joblib.dump(best_model, f"{base_path}/rf_model.pkl")
print("✅ Best model saved as rf_model.pkl")


In [ ]:
# Feature importance plot

importances = best_model.feature_importances_
indices = np.argsort(importances)[-10:]

plt.figure(figsize=(7,5))
plt.barh(range(len(indices)), importances[indices], align='center')
plt.yticks(range(len(indices)), [feature_cols[i] for i in indices])
plt.title('Top 10 Feature Importances')
plt.show()


###A1

In [ ]:
pd.DataFrame(X_new[:5]).head()

# S2

## Part -1 TF-IDF Vectorisation

In [ ]:
!pip install -U sentence-transformers


In [ ]:
# Cell 1: Load cleaned dataset and feature list
import os, numpy as np, pandas as pd, joblib

# adjust this path if you saved elsewhere
base_path = "/content/drive/MyDrive/Phishing_Project/data"
csv_path = os.path.join(base_path, "phish_dataset.csv")

# If file not in drive, adjust to '/content/phish_dataset.csv' or list folder
if not os.path.exists(csv_path):
    raise FileNotFoundError(f"{csv_path} not found. Check path or upload file.")

df = pd.read_csv(csv_path)
print("Loaded dataset shape:", df.shape)
print("Columns:", df.columns.tolist()[:20])

# Determine numeric feature columns (everything except url and label)
feature_cols = [c for c in df.columns if c not in ['url','label']]
print("Number of numeric features:", len(feature_cols))
print("Sample numeric features:", feature_cols[:10])


In [ ]:
# Cell 2: Compute TF-IDF on character n-grams for the URL column
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

urls = df['url'].astype(str).tolist()
print("Number of URLs:", len(urls))

# Character n-grams capture substrings and obfuscation patterns well.
# ngram_range=(3,6) is a good balance for URLs (tweak later if needed)
tfidf = TfidfVectorizer(analyzer='char_wb', ngram_range=(3,6), max_features=4096)  # 4k features
X_text = tfidf.fit_transform(urls)  # sparse matrix
print("TF-IDF text matrix shape:", X_text.shape)

# Save vectorizer for reproducibility
joblib.dump(tfidf, os.path.join(base_path, "tfidf_char_ngram_3_6.joblib"))
print("Saved TF-IDF vectorizer.")


In [ ]:
# Cell 3: Prepare numeric matrix and merge with TF-IDF features
import scipy.sparse as sp

# Numeric features matrix (we previously scaled; if not, we do a quick scaler now)
X_numeric = df[feature_cols].fillna(0).to_numpy(dtype=float)  # shape (N, F)
print("Numeric matrix shape:", X_numeric.shape)

# Optionally scale numeric features
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_num_scaled = scaler.fit_transform(X_numeric)
joblib.dump(scaler, os.path.join(base_path, "scaler_numeric.joblib"))

# Convert numeric dense to sparse and hstack with TF-IDF
X_num_sparse = sp.csr_matrix(X_num_scaled)
X_hybrid = sp.hstack([X_text, X_num_sparse], format='csr')
print("Hybrid feature matrix shape:", X_hybrid.shape)

# Labels
y = df['label'].astype(int).to_numpy()

# Train-test split (stratify to preserve class balance)
X_train, X_test, y_train, y_test = train_test_split(X_hybrid, y, test_size=0.25, random_state=42, stratify=y)
print("Train shape:", X_train.shape, "Test shape:", X_test.shape)

# Save quick numpy references if needed (large sparse => use joblib)
joblib.dump((X_train, X_test, y_train, y_test), os.path.join(base_path, "hybrid_data_joblib.pkl"))
print("Saved hybrid training/test data.")


In [ ]:
# Cell 4: Train RandomForest on hybrid features
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import time

rf = RandomForestClassifier(n_estimators=200, max_depth=18, n_jobs=-1, random_state=42)

t0 = time.time()
rf.fit(X_train, y_train)
t1 = time.time()
print(f"Training time: {t1-t0:.1f}s")

# Predict & evaluate
y_pred = rf.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, digits=4))

# Confusion matrix plot
ConfusionMatrixDisplay.from_predictions(y_test, y_pred)
plt.title("RandomForest - Hybrid TFIDF + Numeric")
plt.show()

# Save model
joblib.dump(rf, os.path.join(base_path, "rf_hybrid_tfidf_numeric.pkl"))
print("Saved RF model.")


In [ ]:
# Cell 5: Ablation to show benefit of hybrid approach
# Extract parts: text-only, numeric-only from saved X_train/X_test (they are sparse Hstack)
# We previously stacked as [text, numeric]; tfidf max_features=4096 so numeric starts at col index 4096
text_dim = X_text.shape[1]
print("Text dim:", text_dim)

# Helper to slice sparse matrices
X_train_text = X_train[:, :text_dim]
X_train_num  = X_train[:, text_dim:]
X_test_text  = X_test[:, :text_dim]
X_test_num   = X_test[:, text_dim:]

from sklearn.ensemble import RandomForestClassifier
def eval_model(Xtr, Xte):
    m = RandomForestClassifier(n_estimators=150, n_jobs=-1, random_state=42)
    m.fit(Xtr, y_train)
    p = m.predict(Xte)
    from sklearn.metrics import f1_score, accuracy_score
    return accuracy_score(y_test, p), f1_score(y_test, p)

print("Training/eval, this may take a few minutes...")
acc_text, f1_text = eval_model(X_train_text, X_test_text)
acc_num,  f1_num  = eval_model(X_train_num,  X_test_num)
acc_hyb,  f1_hyb  = accuracy_score(y_test, y_pred), f1_score(y_test, y_pred)

print(f"Text-only  -> Acc={acc_text:.4f}, F1={f1_text:.4f}")
print(f"Numeric-only -> Acc={acc_num:.4f}, F1={f1_num:.4f}")
print(f"Hybrid -> Acc={acc_hyb:.4f}, F1={f1_hyb:.4f}")


In [ ]:
pip install torch==2.2.2 transformers==4.43.3 sentence-transformers==2.6.1


In [ ]:
# --- Load dataset again ---
import pandas as pd
import os

base_path = "/content/drive/MyDrive/Phishing_Project/data"
df = pd.read_csv(f"{base_path}/phish_dataset.csv")
print("✅ Dataset loaded, rows:", len(df))
print("Columns:", df.columns.tolist()[:10])

# --- Extract URL list ---
urls = df['url'].astype(str).tolist()
print("✅ URLs ready for embedding. Example:")
print(urls[:3])

# --- Generate MiniLM embeddings ---
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("all-MiniLM-L6-v2")

emb = model.encode(urls, batch_size=32, show_progress_bar=True)
print("✅ Embeddings generated. Shape:", emb.shape)

# --- Save for reuse ---
import numpy as np
np.save(f"{base_path}/minilm_embeddings.npy", emb)
print("✅ Saved embeddings as minilm_embeddings.npy")

In [ ]:
import pandas as pd
import os

base_path = "/content/drive/MyDrive/Phishing_Project/data"
df = pd.read_csv(f"{base_path}/phish_dataset.csv")

urls = df['url'].astype(str).tolist()
print("Total URLs:", len(urls))


In [ ]:
import numpy as np
from tqdm import tqdm

embeddings = model.encode(urls, batch_size=32, show_progress_bar=True)
print("✅ Embeddings shape:", embeddings.shape)


In [ ]:
np.save(f"{base_path}/bert_embeddings.npy", embeddings)
print("✅ Embeddings saved as bert_embeddings.npy")


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# Pick 3 random URLs
test_urls = df['url'].sample(3, random_state=42).tolist()
emb_test = model.encode(test_urls)

print("Sample URLs:")
for u in test_urls: print("-", u)

print("\nPairwise similarity:")
print(cosine_similarity(emb_test))


## Part-2 MiniLM Embedding

In [ ]:
#loading all resources
import os, numpy as np, pandas as pd, joblib

base_path = "/content/drive/MyDrive/Phishing_Project/data"

# Load dataset (for numeric features + labels)
df = pd.read_csv(f"{base_path}/phish_dataset.csv")

# Load MiniLM embeddings
emb = np.load(f"{base_path}/minilm_embeddings.npy")
print("✅ Embeddings shape:", emb.shape)

# Separate numeric features and labels
feature_cols = [c for c in df.columns if c not in ['url', 'label']]
X_num = df[feature_cols].fillna(0).to_numpy(dtype=float)
y = df['label'].astype(int).to_numpy()
print("✅ Numeric features shape:", X_num.shape, "Labels:", y.shape)


In [ ]:
#scaling numerical features
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_num_scaled = scaler.fit_transform(X_num)
joblib.dump(scaler, f"{base_path}/scaler_hybrid.pkl")
print("✅ Numeric features scaled")
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_num_scaled = scaler.fit_transform(X_num)
joblib.dump(scaler, f"{base_path}/scaler_hybrid.pkl")
print("✅ Numeric features scaled")


In [ ]:
#Concatenate embeddings + numeric features

X_hybrid = np.hstack([emb, X_num_scaled])
print("✅ Hybrid feature matrix shape:", X_hybrid.shape)


In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X_hybrid, y, test_size=0.25, random_state=42, stratify=y
)
print("✅ Train/Test sizes:", X_train.shape, X_test.shape)


In [ ]:
# Training Multiple models

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import time

# Logistic Regression
t0 = time.time()
lr = LogisticRegression(max_iter=1000, n_jobs=-1)
lr.fit(X_train, y_train)
pred_lr = lr.predict(X_test)
print("\n🔹 Logistic Regression")
print("Accuracy:", accuracy_score(y_test, pred_lr))
print("F1-score:", f1_score(y_test, pred_lr))
print(classification_report(y_test, pred_lr, digits=4))
print("Time:", round(time.time()-t0,2), "s")

# Random Forest
t0 = time.time()
rf = RandomForestClassifier(n_estimators=200, max_depth=15, n_jobs=-1, random_state=42)
rf.fit(X_train, y_train)
pred_rf = rf.predict(X_test)
print("\n🔹 Random Forest")
print("Accuracy:", accuracy_score(y_test, pred_rf))
print("F1-score:", f1_score(y_test, pred_rf))
print(classification_report(y_test, pred_rf, digits=4))
print("Time:", round(time.time()-t0,2), "s")


In [ ]:
#Confusion Matrix Visualization

ConfusionMatrixDisplay.from_predictions(y_test, pred_rf)
plt.title("Random Forest – MiniLM + Numeric Hybrid")
plt.show()
ConfusionMatrixDisplay.from_predictions(y_test, pred_rf)
plt.title("Random Forest – MiniLM + Numeric Hybrid")
plt.show()
ConfusionMatrixDisplay.from_predictions(y_test, pred_rf)
plt.title("Random Forest – MiniLM + Numeric Hybrid")
plt.show()


In [ ]:
#saving model
joblib.dump(rf, f"{base_path}/rf_hybrid_minilm.pkl")
print("✅ Hybrid Random Forest model saved")


In [ ]:
importances = rf.feature_importances_
top_idx = np.argsort(importances)[-15:]
plt.barh(range(15), importances[top_idx], align='center')
plt.yticks(range(15), [f"f{i}" for i in top_idx])
plt.title("Top 15 Hybrid Feature Importances")
plt.show()


# S3



## Part-1 Rulescore computation

In [ ]:
# Define Security Rules
import re
import tldextract

# Suspicious TLDs often seen in phishing
SUSPICIOUS_TLDS = [
    "tk", "ml", "ga", "cf", "gq", "xyz", "top", "club", "work", "zip", "link", "cn","dk"
]

# Common suspicious keywords
PHISHING_KEYWORDS = [
    "login", "verify", "update", "secure", "account", "bank", "payment", "signin", "confirm"
]

# URL shorteners
URL_SHORTENERS = [
    "bit.ly", "tinyurl", "goo.gl", "t.co", "is.gd", "buff.ly", "adf.ly", "shorturl", "ow.ly", "tiny.one", "azure"
]

def compute_rule_score(url):
    score = 0
    rules_triggered = []

    # Rule 1: Suspicious TLD
    ext = tldextract.extract(url)
    if ext.suffix in SUSPICIOUS_TLDS:
        score += 1; rules_triggered.append("Suspicious_TLD")

    # Rule 2: '@' symbol
    if "@" in url:
        score += 1; rules_triggered.append("@_symbol")

    # Rule 3: Too many subdomains
    if url.count(".") > 4:
        score += 1; rules_triggered.append("Excess_subdomains")

    # Rule 4: Contains phishing keyword
    if any(k in url.lower() for k in PHISHING_KEYWORDS):
        score += 1; rules_triggered.append("Keyword_match")

    # Rule 5: URL shortening service
    if any(short in url.lower() for short in URL_SHORTENERS):
        score += 1; rules_triggered.append("Shortener")

    # Normalize (max 5 rules)
    normalized_score = score / 5
    return normalized_score, rules_triggered


In [ ]:
# Apply Rules to Dataset

# Load dataset again
import pandas as pd, os
base_path = "/content/drive/MyDrive/Phishing_Project/data"
df = pd.read_csv(f"{base_path}/phish_dataset.csv")

# Apply rule engine
df['rule_score'] = 0.0
df['triggered_rules'] = ""

for i, url in enumerate(df['url']):
    s, r = compute_rule_score(url)
    df.at[i, 'rule_score'] = s
    df.at[i, 'triggered_rules'] = ",".join(r)

print("✅ Rule scores generated")
print(df[['url', 'rule_score', 'triggered_rules']].head())

# Save for later use
df.to_csv(f"{base_path}/phish_with_rules.csv", index=False)
print("✅ Saved rule-augmented dataset")


In [ ]:
#Integrate with Hybrid ML Model → Trust Index

import numpy as np, joblib

# Load hybrid model & embeddings
rf = joblib.load(f"{base_path}/rf_hybrid_minilm.pkl")
emb = np.load(f"{base_path}/minilm_embeddings.npy")

# Reload numeric features
feature_cols = [c for c in df.columns if c not in ['url', 'label', 'rule_score', 'triggered_rules']]
X_num = df[feature_cols].fillna(0).to_numpy(dtype=float)

# Scale numeric part
scaler = joblib.load(f"{base_path}/scaler_hybrid.pkl")
X_num_scaled = scaler.transform(X_num)

# Combine MiniLM + numeric
X_hybrid = np.hstack([emb, X_num_scaled])

# ML prediction probabilities
proba = rf.predict_proba(X_hybrid)[:,1]

# Combine with rule score
trust_index = (0.7 * proba) + (0.3 * (1 - df['rule_score'].to_numpy()))
df['Trust_Index'] = trust_index

print("✅ Trust Index computed")
print(df[['url','label','rule_score','Trust_Index']].head())

# Save output
df.to_csv(f"{base_path}/phish_with_trust_index.csv", index=False)
print("✅ Saved phish_with_trust_index.csv")


## Part-2 Analyse Risk Level from rule score

In [ ]:
# Loading Results and Defining ThresholdsLoad Results and Define Thresholds

import pandas as pd
import os

base_path = "/content/drive/MyDrive/Phishing_Project/data"
file_path = os.path.join(base_path, "phish_with_trust_index.csv")

df = pd.read_csv(file_path)
print("✅ Data loaded with columns:", df.columns.tolist()[:10])
print("Rows:", len(df))


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/Phishing_Project/data/phish_with_trust_index.csv'

In [ ]:
def classify_risk(trust_index):
    if trust_index >= 0.70:
        return "Safe"
    elif trust_index >= 0.40:
        return "Suspicious"
    else:
        return "Phishing"

df['Risk_Level'] = df['Trust_Index'].apply(classify_risk)
print(df[['url','Trust_Index','rule_score','Risk_Level']].head())


In [ ]:
#Explainability layer

def explain_triage(row):
    ml_score = row['Trust_Index']
    rule_score = row['rule_score']
    rules = row['triggered_rules']

    if row['Risk_Level'] == "Phishing":
        reason = f"High rule_score ({rule_score:.2f}); rules triggered: {rules or 'None'}"
    elif row['Risk_Level'] == "Suspicious":
        reason = f"Mixed indicators: ML={ml_score:.2f}, rule={rule_score:.2f}"
    else:
        reason = f"Low risk: ML={ml_score:.2f}, rule={rule_score:.2f}"

    return reason

df['Triage_Explanation'] = df.apply(explain_triage, axis=1)
print(df[['url','Risk_Level','Triage_Explanation']].head())


In [ ]:
# Summary statistics

summary = df['Risk_Level'].value_counts(normalize=True) * 100
print("🔹 Risk Distribution (%):\n", summary.round(2))

# Save final triage output
out_path = os.path.join(base_path, "phish_final_triage.csv")
df.to_csv(out_path, index=False)
print(f"✅ Final triage file saved at: {out_path}")


In [ ]:
# Visualization

summary = df['Risk_Level'].value_counts(normalize=True) * 100
print("🔹 Risk Distribution (%):\n", summary.round(2))

# Save final triage output
out_path = os.path.join(base_path, "phish_final_triage.csv")
df.to_csv(out_path, index=False)
print(f"✅ Final triage file saved at: {out_path}")



## Part-3 Visual analysis of Outputs
(Pie Chart, Histogram, etc)

In [ ]:
# Load final triage data

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

base_path = "/content/drive/MyDrive/Phishing_Project/data"
df = pd.read_csv(f"{base_path}/phish_final_triage.csv")

print("✅ Data loaded for visualization")
print(df[['url','Trust_Index','Risk_Level']].head())
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

base_path = "/content/drive/MyDrive/Phishing_Project/data"
df = pd.read_csv(f"{base_path}/phish_final_triage.csv")

print("✅ Data loaded for visualization")
print(df[['url','Trust_Index','Risk_Level']].head())


In [ ]:
#Risk Distribution Visuals

# Pie chart
plt.figure(figsize=(6,6))
df['Risk_Level'].value_counts().plot(kind='pie', autopct='%1.1f%%',
                                     colors=['limegreen','gold','tomato'],
                                     shadow=True)
plt.title("Phishing URL Risk Distribution")
plt.ylabel("")
plt.show()

# Bar chart
plt.figure(figsize=(6,4))
sns.countplot(x='Risk_Level', data=df, palette=['limegreen','gold','tomato'])
plt.title("Risk Category Counts")
plt.xlabel("Risk Level")
plt.ylabel("Count")
plt.show()


In [ ]:
#Trust index trend

plt.figure(figsize=(8,4))
sns.histplot(df['Trust_Index'], bins=30, kde=True, color='skyblue')
plt.title("Distribution of Trust Index Across URLs")
plt.xlabel("Trust Index (0–1)")
plt.ylabel("Frequency")
plt.show()



In [ ]:
#Most Triggered Rules (for novelty visualization)

from collections import Counter

all_rules = ",".join(df['triggered_rules'].dropna().tolist()).split(',')
rule_counts = Counter([r for r in all_rules if r])

plt.figure(figsize=(7,4))
sns.barplot(x=list(rule_counts.keys()), y=list(rule_counts.values()), palette="viridis")
plt.title("Top Triggered Rules in Phishing URLs")
plt.xlabel("Rule Name")
plt.ylabel("Count")
plt.xticks(rotation=30)
plt.show()


In [ ]:
#exporting
summary = df['Risk_Level'].value_counts(normalize=True).mul(100).round(2)
summary.to_csv(f"{base_path}/risk_summary.csv")
print("✅ Summary CSV saved as risk_summary.csv")
print(summary)


# S4

## Part - 1 Testing models

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# base project directory
base_path = "/content/drive/MyDrive/Phishing_Project/data"
print("✅ Drive mounted and base path set:", base_path)

In [ ]:
!pip install -q sentence-transformers==3.0.1 scikit-learn==1.5.2 joblib==1.4.2 streamlit==1.39.0 tldextract
import sklearn, joblib, streamlit, sentence_transformers

In [ ]:
import os, joblib, numpy as np

# list all files to confirm
os.listdir(base_path)

In [ ]:
#loadinng
rf_model = joblib.load(f"{base_path}/rf_hybrid_minilm.pkl")
scaler = joblib.load(f"{base_path}/scaler_hybrid.pkl")
print("✅ Model and scaler loaded successfully!")


In [ ]:
#Load MiniLM Model (Sentence-BERT)

from sentence_transformers import SentenceTransformer
minilm_model = SentenceTransformer("all-MiniLM-L6-v2")
print("✅ MiniLM model loaded successfully!")



In [ ]:
# Test a Single MiniLM Embedding

sample = ["https://paypal-login.tk", "https://www.google.com"]
emb = minilm_model.encode(sample, show_progress_bar=False)
print("✅ Embedding shape:", emb.shape)


In [ ]:
# Test Numeric Scaler

import pandas as pd
df_test = pd.read_csv(f"{base_path}/phish_dataset.csv").head(5)
feature_cols = [c for c in df_test.columns if c not in ['url','label']]
num_scaled = scaler.transform(df_test[feature_cols])
print("✅ Numeric scaler working; sample shape:", num_scaled.shape)


In [ ]:
# Confirm Rule Engine Availability

import re, tldextract
SUSPICIOUS_TLDS = ["tk","ml","ga","cf","gq","xyz","top","club","work","zip","link","cn"]
PHISHING_KEYWORDS = ["login","verify","update","secure","account","bank","payment","signin","confirm"]
URL_SHORTENERS = ["bit.ly","tinyurl","goo.gl","t.co","is.gd","buff.ly","adf.ly","shorturl"]

def compute_rule_score(url):
    score, rules = 0, []
    ext = tldextract.extract(url)
    if ext.suffix in SUSPICIOUS_TLDS: score += 1; rules.append("Suspicious_TLD")
    if "@" in url: score += 1; rules.append("@_symbol")
    if url.count(".") > 4: score += 1; rules.append("Excess_subdomains")
    if any(k in url.lower() for k in PHISHING_KEYWORDS): score += 1; rules.append("Keyword_match")
    if any(short in url.lower() for short in URL_SHORTENERS): score += 1; rules.append("Shortener")
    return score/5, rules

print("✅ Rule engine ready.")


In [ ]:
# End-to-End Sanity Check

test_url = "https://paypal-login.tk"
# 1. get embedding
emb_vec = minilm_model.encode([test_url])
# 2. get numeric placeholder (dummy small vector)
import numpy as np
num_vec = np.zeros((1, len(scaler.mean_)))  # same shape as numeric features
num_scaled = scaler.transform(num_vec)
# 3. concat
X_test = np.hstack([emb_vec, num_scaled])
# 4. predict
prob = rf_model.predict_proba(X_test)[0][1]
rule_score, rules = compute_rule_score(test_url)
trust_index = 0.7*prob + 0.3*(1-rule_score)
print(f"ML_Prob={prob:.3f}, Rule={rule_score:.3f}, TrustIndex={trust_index:.3f}, Rules={rules}")


## Part-2 Import & clean dataset, extract features, S3 P1

In [ ]:
# helper imports + load metadata (feature list & means)


import pandas as pd, numpy as np, os, joblib

# base_path = "/content/drive/MyDrive/Phishing_Project/data"

# load training dataset to get feature names & means
train_df = pd.read_csv(os.path.join(base_path, "phish_dataset.csv"))

# feature columns (numeric handcrafted features used in training)
feature_cols = [c for c in train_df.columns if c not in ['url','label','rule_score','triggered_rules','Trust_Index','Risk_Level','Triage_Explanation']]
print("Number of numeric feature columns expected by scaler:", len(feature_cols))

# compute mean values (used to fill features we don't compute right now)
feature_means = train_df[feature_cols].mean().to_dict()

# load scaler and model (ensure these files exist)
scaler = joblib.load(os.path.join(base_path, "scaler_hybrid.pkl"))
rf_model = joblib.load(os.path.join(base_path, "rf_hybrid_minilm.pkl"))

print("Loaded scaler and RF model. Example feature names:", feature_cols[:12])


In [ ]:
# Cell B — basic numeric feature extractor for a single URL
import re
import tldextract
from urllib.parse import urlparse

# list of URL shorteners (same as rule engine)
URL_SHORTENERS = ["bit.ly","tinyurl","goo.gl","t.co","is.gd","buff.ly","adf.ly","shorturl","ow.ly","tiny.one"]

def extract_basic_numeric_features(url: str):
    # initialize dict
    out = {}
    u = str(url).strip()
    if not (u.startswith("http://") or u.startswith("https://")):
        # add http so urlparse works consistently
        u_parse = urlparse("http://" + u)
    else:
        u_parse = urlparse(u)
    host = u_parse.netloc
    path = u_parse.path or ""
    query = u_parse.query or ""
    full = u

    # core simple features (these map to many common feature names)
    out['length_url'] = len(full)
    out['length_hostname'] = len(host)
    out['ip'] = 1 if re.match(r'^\d{1,3}(\.\d{1,3}){3}$', host.split(':')[0]) else 0
    out['nb_dots'] = full.count('.')
    out['nb_hyphens'] = full.count('-')
    out['nb_at'] = full.count('@')
    out['nb_qm'] = full.count('?')
    out['nb_and'] = full.count('&')
    out['nb_or'] = full.count('|')
    out['nb_eq'] = full.count('=')
    out['nb_underscore'] = full.count('_')
    out['nb_tilde'] = full.count('~')
    out['nb_percent'] = full.count('%')
    out['nb_slash'] = full.count('/')
    out['nb_star'] = full.count('*')
    out['nb_colon'] = full.count(':')
    out['nb_comma'] = full.count(',')
    out['nb_semicolumn'] = full.count(';')
    out['nb_dollar'] = full.count('$')
    out['nb_space'] = full.count(' ')
    out['nb_www'] = 1 if 'www.' in full.lower() else 0
    out['nb_com'] = 1 if '.com' in full.lower() else 0
    out['nb_dslash'] = full.count('//')
    out['http_in_path'] = 1 if 'http' in path.lower() else 0
    out['https_token'] = 1 if 'https' in full.lower() else 0
    # ratio digits
    digits_in_url = sum(c.isdigit() for c in full)
    out['ratio_digits_url'] = digits_in_url / len(full) if len(full)>0 else 0.0
    digits_in_host = sum(c.isdigit() for c in host)
    out['ratio_digits_host'] = digits_in_host / len(host) if len(host)>0 else 0.0
    # punycode
    out['punycode'] = 1 if 'xn--' in host.lower() else 0
    # port
    out['port'] = 1 if ':' in host and host.split(':')[-1].isdigit() else 0
    # tld presence in path/subdomain
    ext = tldextract.extract(full)
    out['tld'] = ext.suffix or ''
    out['tld_in_path'] = 1 if out['tld'] and out['tld'] in path.lower() else 0
    out['tld_in_subdomain'] = 1 if out['tld'] and out['tld'] in ext.subdomain.lower() else 0
    out['nb_subdomains'] = len(ext.subdomain.split('.')) if ext.subdomain else 0
    out['shortening_service'] = 1 if any(s in full.lower() for s in URL_SHORTENERS) else 0
    # path extension
    out['path_extension'] = 1 if re.search(r'\.[a-zA-Z0-9]{1,5}$', path) else 0
    # basic words metrics
    words = re.findall(r'[A-Za-z0-9]+', full)
    out['length_words_raw'] = sum(len(w) for w in words)
    out['char_repeat'] = max((max([u.count(k) for k in set(full)]) if full else 0), 0)
    out['shortest_words_raw'] = min((len(w) for w in words), default=0)
    out['longest_words_raw'] = max((len(w) for w in words), default=0)
    out['avg_words_raw'] = (sum(len(w) for w in words) / len(words)) if words else 0
    # many other features exist in training data — not implemented here (will be filled from means)
    return out


In [ ]:
# Cell C — complete predict_url() function
import numpy as np
from typing import Dict

# load minilm model and rule function if not already loaded in session
# (assumes minilm_model, rf_model, scaler, feature_cols, feature_means, compute_rule_score are in scope)

def predict_url(url: str) -> Dict:
    url = str(url).strip()
    # 1) compute basic numeric features we implemented
    basic = extract_basic_numeric_features(url)

    # 2) create full feature vector in the exact order of feature_cols
    num_vec = []
    missing_features = []
    for f in feature_cols:
        if f in basic:
            val = basic[f]
        else:
            # if feature not computed, use mean from training
            val = feature_means.get(f, 0.0)
            missing_features.append(f)
        # ensure numeric type
        try:
            num_vec.append(float(val))
        except:
            num_vec.append(0.0)
    num_vec = np.array(num_vec).reshape(1, -1)  # shape (1, n_features)

    # 3) scale numeric features
    num_vec_scaled = scaler.transform(num_vec)

    # 4) compute embedding for the url (MiniLM)
    emb_vec = minilm_model.encode([url], show_progress_bar=False)
    # ensure emb_vec shape (1, 384)

    # 5) concat to hybrid feature vector
    X_hybrid = np.hstack([emb_vec, num_vec_scaled])

    # 6) ML probability from RF
    ml_prob = float(rf_model.predict_proba(X_hybrid)[0][1])

    # 7) rule score & triggered rules
    rule_score, triggered_rules = compute_rule_score(url)

    # 8) Trust Index (tunable weights)
    TRUST_WEIGHT_ML = 0.7
    TRUST_WEIGHT_RULE = 0.3
    trust_index = (TRUST_WEIGHT_ML * ml_prob) + (TRUST_WEIGHT_RULE * (1 - rule_score))
    # clamp to [0,1]
    trust_index = max(0.0, min(1.0, trust_index))

    # 9) risk level thresholds (customizable)
    if trust_index >= 0.70:
        risk = "Safe"
    elif trust_index >= 0.40:
        risk = "Suspicious"
    else:
        risk = "Phishing"

    # 10) explanation text
    explanation = (f"ML_prob={ml_prob:.3f}; rule_score={rule_score:.3f}; "
                   f"rules_triggered={'|'.join(triggered_rules) if triggered_rules else 'None'}; "
                   f"filled_features_count={len(missing_features)}")

    # 11) return a dict (easy to use in UI or logs)
    result = {
        "url": url,
        "ml_prob": ml_prob,
        "rule_score": rule_score,
        "triggered_rules": triggered_rules,
        "trust_index": trust_index,
        "risk_level": risk,
        "explanation": explanation,
        "numeric_features_filled_from_mean": missing_features
    }
    return result

# Quick example
print(predict_url("https://paypal-login.tk"))


In [ ]:
import csv, datetime

log_path = os.path.join(base_path, "live_prediction_logs.csv")
def log_prediction(res_dict):
    header = ["timestamp","url","ml_prob","rule_score","trust_index","risk_level","triggered_rules","explanation"]
    write_header = not os.path.exists(log_path)
    with open(log_path, "a", newline="") as f:
        writer = csv.writer(f)
        if write_header:
            writer.writerow(header)
        writer.writerow([
            datetime.datetime.utcnow().isoformat(),
            res_dict["url"],
            res_dict["ml_prob"],
            res_dict["rule_score"],
            res_dict["trust_index"],
            res_dict["risk_level"],
            ";".join(res_dict["triggered_rules"]),
            res_dict["explanation"]
        ])
    return log_path

# Example
r = predict_url("https://paypal-login.tk")
print("Logged at:", log_prediction(r))


## Part-3 Streamlit App and SHAP XAI

In [ ]:
!pip install pyngrok


In [ ]:
#Create a Streamlit App File

In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
import time
from datetime import datetime

# import the predict_url and log_prediction functions from your Colab session
# (since we're running inline, we'll redefine a lightweight wrapper)
from sentence_transformers import SentenceTransformer
import joblib, os, re, tldextract
from urllib.parse import urlparse

base_path = "/content/drive/MyDrive/Phishing_Project/data"

# --- Load Model and Scaler ---
rf_model = joblib.load(os.path.join(base_path, "rf_hybrid_minilm.pkl"))
scaler = joblib.load(os.path.join(base_path, "scaler_hybrid.pkl"))
minilm_model = SentenceTransformer("all-MiniLM-L6-v2")

# --- Rule Engine (reuse from earlier) ---
SUSPICIOUS_TLDS = ["tk","ml","ga","cf","gq","xyz","top","club","work","zip","link","cn"]
PHISHING_KEYWORDS = ["login","verify","update","secure","account","bank","payment","signin","confirm"]
URL_SHORTENERS = ["bit.ly","tinyurl","goo.gl","t.co","is.gd","buff.ly","adf.ly","shorturl"]

def compute_rule_score(url):
    score, rules = 0, []
    ext = tldextract.extract(url)
    if ext.suffix in SUSPICIOUS_TLDS: score += 1; rules.append("Suspicious_TLD")
    if "@" in url: score += 1; rules.append("@_symbol")
    if url.count(".") > 4: score += 1; rules.append("Excess_subdomains")
    if any(k in url.lower() for k in PHISHING_KEYWORDS): score += 1; rules.append("Keyword_match")
    if any(short in url.lower() for short in URL_SHORTENERS): score += 1; rules.append("Shortener")
    return score/5, rules

# --- Basic Numeric Feature Extractor (simplified for UI) ---
def extract_basic_features(url):
    u = str(url)
    return {
        "length_url": len(u),
        "nb_dots": u.count("."),
        "nb_hyphens": u.count("-"),
        "https_token": 1 if "https" in u.lower() else 0,
        "ratio_digits_url": sum(c.isdigit() for c in u)/len(u) if len(u)>0 else 0,
    }

# --- Mini Predict Function (simplified for UI) ---
def predict_url_simple(url):
    emb = minilm_model.encode([url], show_progress_bar=False)
    # numeric placeholder
    feat = np.array(list(extract_basic_features(url).values())).reshape(1,-1)
    # scale dummy numeric with same dimension
    pad = np.zeros((1, scaler.mean_.shape[0]-feat.shape[1]))
    num_scaled = np.hstack([feat, pad])
    # ML
    prob = rf_model.predict_proba(np.hstack([emb, num_scaled]))[0][1]
    rule_score, rules = compute_rule_score(url)
    trust_index = 0.7*prob + 0.3*(1-rule_score)
    trust_index = max(0.0, min(1.0, trust_index))
    if trust_index >= 0.70: risk="Safe"
    elif trust_index >= 0.40: risk="Suspicious"
    else: risk="Phishing"
    return prob, rule_score, trust_index, risk, rules

# --- Streamlit UI ---
st.set_page_config(page_title="PhishTriage – URL Detector", page_icon="🛡️", layout="centered")

st.title("🛡️ PhishTriage – Real-Time Phishing URL Detector")
st.markdown("Enter any URL below to analyze its risk level using our hybrid AI + Cybersecurity model.")

url_input = st.text_input("🔗 Enter URL here:", placeholder="https://example.com")
check_btn = st.button("Check URL")

if check_btn and url_input.strip():
    with st.spinner("Analyzing... Please wait"):
        prob, rule_score, trust_index, risk, rules = predict_url_simple(url_input)
        time.sleep(1.5)
    st.subheader("Results:")
    color = {"Safe":"green","Suspicious":"orange","Phishing":"red"}[risk]
    st.markdown(f"**Risk Level:** <span style='color:{color};font-size:22px'><b>{risk}</b></span>", unsafe_allow_html=True)
    st.progress(int(trust_index*100))
    st.write(f"**Trust Index:** {trust_index:.3f}")
    st.write(f"**Model Probability (Phishing):** {prob:.3f}")
    st.write(f"**Rule Score:** {rule_score:.3f}")
    st.write(f"**Triggered Rules:** {', '.join(rules) if rules else 'None'}")
    st.write(f"Checked at: {datetime.now().strftime('%Y-%m-d %H:%M:%S')}")
    st.success("✅ Analysis completed successfully!")
else:
    st.info("Enter a URL and click **Check URL** to begin.")


In [ ]:
!streamlit run app.py --server.port 8501 --server.address 0.0.0.0


In [ ]:
#part -4 : Live Integration & TestingLive Integration & Testing

In [ ]:
# Continued in VS Code Streamlit app

# S5

In [ ]:
pip install shap==0.46.0 matplotlib==3.9.2 seaborn==0.13.2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 543.9/543.9 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 104.6 MB/s eta 0:00:00
  Attempting uninstall: matplotlib
    Found existing installation: matplotlib 3.10.0
    Uninstalling matplotlib-3.10.0:
      Successfully uninstalled matplotlib-3.10.0
  Attempting uninstall: shap
    Found existing installation: shap 0.49.1
    Uninstalling shap-0.49.1:
      Successfully uninstalled shap-0.49.1


In [ ]:
# Continued VS code as explain_setup.py